# 3-RRR Spherical Parallel Mechanism - Symbolic Kinematics

**Author:** Chawin Ophaswongse  
**Date:** January 2026  
**Purpose:** Symbolic derivation and verification of forward/inverse kinematics

---

This notebook uses SymPy for symbolic computation to:
1. Derive forward kinematics equations
2. Verify the C++ implementation formulas
3. Visualize mechanism configurations
4. Generate test cases with known solutions

## 1. Setup and Imports

In [ ]:
import sympy as sp
from sympy import symbols, cos, sin, Matrix, sqrt, simplify, trigsimp
from sympy import pi, atan2, acos, asin
from sympy.vector import CoordSys3D
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Enable pretty printing
sp.init_printing(use_unicode=True)

print("SymPy version:", sp.__version__)
print("NumPy version:", np.__version__)

## 2. Define Symbolic Variables

### Architecture Parameters
- $\alpha_{i1}$: Proximal link angle (angle from input axis to intermediate joint)
- $\alpha_{i2}$: Distal link angle (angle from intermediate joint to output)
- $\beta_{i1}$: Base joint orientation angles
- $\beta_{i2}$: Moving platform orientation angles

### Joint Variables
- $\theta_i$: Input joint angles (actuated)

In [ ]:
# Architecture parameters (symbolic)
alpha_11, alpha_12, alpha_13 = symbols('alpha_11 alpha_12 alpha_13', real=True)
alpha_21, alpha_22, alpha_23 = symbols('alpha_21 alpha_22 alpha_23', real=True)
beta_11, beta_12, beta_13 = symbols('beta_11 beta_12 beta_13', real=True)
beta_21, beta_22, beta_23 = symbols('beta_21 beta_22 beta_23', real=True)

# Joint angles (input variables)
theta_1, theta_2, theta_3 = symbols('theta_1 theta_2 theta_3', real=True)

# Intermediate variables
epsilon_1, epsilon_2, epsilon_3 = symbols('epsilon_1 epsilon_2 epsilon_3', real=True)

# Group for convenience
theta = [theta_1, theta_2, theta_3]
alpha_i1 = [alpha_11, alpha_12, alpha_13]
alpha_i2 = [alpha_21, alpha_22, alpha_23]
beta_i1 = [beta_11, beta_12, beta_13]
beta_i2 = [beta_21, beta_22, beta_23]

print("Joint angles:", theta)
print("Proximal link angles:", alpha_i1)

## 3. Rotation Matrices

Define basic rotation matrices for spherical kinematics.

In [ ]:
def Rx(angle):
    """Rotation matrix about X-axis"""
    return Matrix([
        [1, 0, 0],
        [0, cos(angle), -sin(angle)],
        [0, sin(angle), cos(angle)]
    ])

def Ry(angle):
    """Rotation matrix about Y-axis"""
    return Matrix([
        [cos(angle), 0, sin(angle)],
        [0, 1, 0],
        [-sin(angle), 0, cos(angle)]
    ])

def Rz(angle):
    """Rotation matrix about Z-axis"""
    return Matrix([
        [cos(angle), -sin(angle), 0],
        [sin(angle), cos(angle), 0],
        [0, 0, 1]
    ])

# Display Rz as example
print("Rz(theta):")
Rz(theta_1)

## 4. Rodrigues' Rotation Formula

For rotating a vector $\mathbf{v}$ about axis $\mathbf{k}$ by angle $\theta$:

$$\mathbf{v}_{rot} = \mathbf{v} \cos\theta + (\mathbf{k} \times \mathbf{v}) \sin\theta + \mathbf{k}(\mathbf{k} \cdot \mathbf{v})(1 - \cos\theta)$$

In [ ]:
def rodrigues_rotation(v, k, angle):
    """
    Rotate vector v about axis k by angle using Rodrigues' formula.
    
    Parameters:
    - v: 3x1 vector to rotate
    - k: 3x1 unit axis of rotation
    - angle: rotation angle
    
    Returns:
    - Rotated vector
    """
    v = Matrix(v)
    k = Matrix(k)
    
    # Cross product k x v
    k_cross_v = Matrix([
        k[1]*v[2] - k[2]*v[1],
        k[2]*v[0] - k[0]*v[2],
        k[0]*v[1] - k[1]*v[0]
    ])
    
    # Dot product k . v
    k_dot_v = k[0]*v[0] + k[1]*v[1] + k[2]*v[2]
    
    # Rodrigues' formula
    v_rot = v * cos(angle) + k_cross_v * sin(angle) + k * k_dot_v * (1 - cos(angle))
    
    return simplify(v_rot)

# Test: rotate [1, 0, 0] about z-axis by 90 degrees should give [0, 1, 0]
test_result = rodrigues_rotation([1, 0, 0], [0, 0, 1], pi/2)
print("Rotate [1,0,0] about z by 90°:")
test_result

## 5. Intermediate Joint Axis (v-vector) Computation

**This is the critical formula that needs verification!**

The v-vector represents the direction of the intermediate joint axis after rotating the proximal link by $\theta_i$.

### Current C++ Implementation (BUGGY):
```cpp
v_vectors[i] = {sin_alpha * sin_theta * cos_beta, 
                sin_alpha * sin_theta * sin_beta,
                cos_alpha};
```

### Problem:
When $\theta = 0$ and $\alpha = \pi/2$: produces zero vector!

### Correct Approach:
Use rotation matrix or Rodrigues' formula to transform the default intermediate axis.

In [ ]:
def compute_v_vector_buggy(theta_i, alpha_i, beta_i):
    """
    BUGGY version from C++ code.
    This produces zero vectors when theta=0!
    """
    return Matrix([
        sin(alpha_i) * sin(theta_i) * cos(beta_i),
        sin(alpha_i) * sin(theta_i) * sin(beta_i),
        cos(alpha_i)
    ])

def compute_v_vector_correct(theta_i, alpha_i, beta_i):
    """
    CORRECT version using proper spherical kinematics.
    
    The intermediate joint axis v_i is computed by:
    1. Start with default axis direction based on alpha
    2. Rotate by beta about z-axis (base orientation)
    3. Rotate by theta about the input axis
    """
    # Default intermediate axis direction (when theta=0)
    # This is at angle alpha from the input axis (z-axis assumed)
    v_default = Matrix([
        sin(alpha_i),
        0,
        cos(alpha_i)
    ])
    
    # Input axis (after rotation by beta about z)
    input_axis = Matrix([0, 0, 1])  # Assuming z-axis as base input
    
    # Apply beta rotation (base joint orientation)
    R_beta = Rz(beta_i)
    v_after_beta = R_beta * v_default
    
    # The input axis direction after beta rotation
    input_axis_rotated = R_beta * input_axis
    
    # Rotate v about the input axis by theta
    # Using Rodrigues' formula
    v_final = rodrigues_rotation(v_after_beta, input_axis_rotated, theta_i)
    
    return trigsimp(v_final)

# Compare buggy vs correct for theta=0, alpha=pi/2
print("=== Testing with theta=0, alpha=pi/2, beta=0 ===")
print("\nBuggy version:")
v_buggy = compute_v_vector_buggy(0, pi/2, 0)
print(v_buggy)

print("\nCorrect version:")
v_correct = compute_v_vector_correct(0, pi/2, 0)
print(v_correct)

## 6. Symbolic v-vector Derivation

Let's derive the general formula for the v-vector symbolically.

In [ ]:
# General symbolic derivation
alpha, beta, theta_sym = symbols('alpha beta theta', real=True)

print("General v-vector formula (symbolic):")
v_general = compute_v_vector_correct(theta_sym, alpha, beta)
print("\nv =")
v_general

In [ ]:
# Simplify each component
print("v_x =", trigsimp(v_general[0]))
print("v_y =", trigsimp(v_general[1]))
print("v_z =", trigsimp(v_general[2]))

## 7. Class I: Agile Eye Configuration

For the Agile Eye (Class I with all angles = 90°):
- $\alpha_{i1} = \alpha_{i2} = \pi/2$ for all limbs
- $\beta_{i1}$ = base joint orientations (typically 0°, 120°, 240°)
- $\beta_{i2} = \pi/2$ (orthogonal moving platform)

In [ ]:
# Agile Eye configuration
def agile_eye_v_vectors(theta_1_val, theta_2_val, theta_3_val):
    """
    Compute v-vectors for Agile Eye configuration.
    Base orientations at 0°, 120°, 240°.
    """
    alpha_val = pi/2
    beta_vals = [0, 2*pi/3, 4*pi/3]  # 0°, 120°, 240°
    theta_vals = [theta_1_val, theta_2_val, theta_3_val]
    
    v_vectors = []
    for i in range(3):
        v_i = compute_v_vector_correct(theta_vals[i], alpha_val, beta_vals[i])
        v_vectors.append(trigsimp(v_i))
    
    return v_vectors

# Test with theta = {0, 0, 0}
print("=== Agile Eye v-vectors for theta = {0, 0, 0} ===")
v_vecs = agile_eye_v_vectors(0, 0, 0)
for i, v in enumerate(v_vecs):
    print(f"\nv_{i+1} = {v.T}")

## 8. Numerical Verification

Convert symbolic expressions to numerical values for testing.

In [ ]:
def v_vector_numerical(theta_val, alpha_val, beta_val):
    """
    Numerical computation of v-vector using correct formula.
    """
    # Default intermediate axis direction
    v_default = np.array([
        np.sin(alpha_val),
        0,
        np.cos(alpha_val)
    ])
    
    # Rotation matrix about z by beta
    c_beta, s_beta = np.cos(beta_val), np.sin(beta_val)
    R_beta = np.array([
        [c_beta, -s_beta, 0],
        [s_beta, c_beta, 0],
        [0, 0, 1]
    ])
    
    v_after_beta = R_beta @ v_default
    
    # Input axis (z-axis rotated by beta)
    k = R_beta @ np.array([0, 0, 1])
    
    # Rodrigues' rotation
    c_theta, s_theta = np.cos(theta_val), np.sin(theta_val)
    k_cross_v = np.cross(k, v_after_beta)
    k_dot_v = np.dot(k, v_after_beta)
    
    v_final = v_after_beta * c_theta + k_cross_v * s_theta + k * k_dot_v * (1 - c_theta)
    
    return v_final

# Test cases
print("=== Numerical v-vector tests ===")
test_cases = [
    (0, np.pi/2, 0),           # theta=0, alpha=90°, beta=0°
    (np.pi/6, np.pi/2, 0),     # theta=30°, alpha=90°, beta=0°
    (0, np.pi/2, 2*np.pi/3),   # theta=0, alpha=90°, beta=120°
]

for theta_val, alpha_val, beta_val in test_cases:
    v = v_vector_numerical(theta_val, alpha_val, beta_val)
    print(f"theta={np.degrees(theta_val):6.1f}°, alpha={np.degrees(alpha_val):5.1f}°, beta={np.degrees(beta_val):6.1f}°")
    print(f"  v = [{v[0]:8.4f}, {v[1]:8.4f}, {v[2]:8.4f}]")
    print(f"  |v| = {np.linalg.norm(v):.6f}")
    print()

## 9. Visualization

3D plot of the mechanism configuration.

In [ ]:
def plot_mechanism(theta_vals, alpha_val=np.pi/2, title="3-RRR SPM Configuration"):
    """
    Visualize the mechanism with given joint angles.
    """
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    # Base orientations (0°, 120°, 240°)
    beta_vals = [0, 2*np.pi/3, 4*np.pi/3]
    colors = ['red', 'green', 'blue']
    
    # Plot unit sphere (lightly)
    u = np.linspace(0, 2 * np.pi, 30)
    v = np.linspace(0, np.pi, 20)
    x = np.outer(np.cos(u), np.sin(v))
    y = np.outer(np.sin(u), np.sin(v))
    z = np.outer(np.ones(np.size(u)), np.cos(v))
    ax.plot_surface(x, y, z, alpha=0.1, color='gray')
    
    # Plot each limb
    for i in range(3):
        # Input axis direction
        c_beta, s_beta = np.cos(beta_vals[i]), np.sin(beta_vals[i])
        R_beta = np.array([[c_beta, -s_beta, 0], [s_beta, c_beta, 0], [0, 0, 1]])
        input_axis = R_beta @ np.array([0, 0, 1])
        
        # v-vector
        v_i = v_vector_numerical(theta_vals[i], alpha_val, beta_vals[i])
        
        # Plot input axis
        ax.quiver(0, 0, 0, input_axis[0], input_axis[1], input_axis[2], 
                  color=colors[i], alpha=0.5, arrow_length_ratio=0.1, linewidth=2)
        
        # Plot v-vector (intermediate joint axis)
        ax.quiver(0, 0, 0, v_i[0], v_i[1], v_i[2], 
                  color=colors[i], arrow_length_ratio=0.1, linewidth=3,
                  label=f'Limb {i+1}: θ={np.degrees(theta_vals[i]):.1f}°')
    
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title(title)
    ax.legend()
    ax.set_box_aspect([1,1,1])
    
    plt.tight_layout()
    plt.show()

# Plot with theta = {0, 0, 0}
plot_mechanism([0, 0, 0], title="Home Position (θ = {0, 0, 0})")

In [ ]:
# Plot with different joint angles
plot_mechanism([np.pi/6, np.pi/6, np.pi/6], title="θ = {30°, 30°, 30°}")

## 10. Generate C++ Code

Export the correct formula for use in C++.

In [ ]:
print("""
// CORRECT v-vector computation for C++
// Replace the buggy formula in forward_kinematics.cpp

Vector3 compute_v_vector(double theta, double alpha, double beta) {
    // Default intermediate axis (when theta=0)
    double v_default_x = std::sin(alpha);
    double v_default_y = 0.0;
    double v_default_z = std::cos(alpha);
    
    // Apply beta rotation (Rz)
    double c_beta = std::cos(beta);
    double s_beta = std::sin(beta);
    double vx = c_beta * v_default_x - s_beta * v_default_y;
    double vy = s_beta * v_default_x + c_beta * v_default_y;
    double vz = v_default_z;
    
    // Input axis is [0, 0, 1] (z-axis), which doesn't change under Rz
    // So we rotate v about z-axis by theta using Rodrigues' formula
    // For rotation about z: v_rot = v*cos(theta) + (z x v)*sin(theta) + z*(z.v)*(1-cos(theta))
    // z x v = [-vy, vx, 0]
    // z.v = vz
    
    double c_theta = std::cos(theta);
    double s_theta = std::sin(theta);
    
    double v_rot_x = vx * c_theta - vy * s_theta;
    double v_rot_y = vy * c_theta + vx * s_theta;
    double v_rot_z = vz;  // z-component unchanged for rotation about z
    
    return {v_rot_x, v_rot_y, v_rot_z};
}
""")

## 11. TODO: Further Work

- [ ] Implement full FK equation derivation (quadratic/quartic formulation)
- [ ] Derive IK closed-form solution
- [ ] Add workspace analysis
- [ ] Generate test cases with known FK solutions
- [ ] Compare Class I, II, III architectures
- [ ] Add singularity analysis

In [ ]:
print("Notebook complete! Use these derivations to fix the C++ implementation.")